In [1]:
## IGRA_band_process.py
# Select IGRA data to post-process

# Libraries
import numpy as np
import os as os
from glob import glob
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import sonde_reanalysis_comparison as src
import igra_utils
import igra
import csv
import pickle

In [2]:
## IGRA_analysis.py
# Retrieve and analyze data from IGRA
# Extracts only RS92 profiles within start_year to end_year (inclusive)

# Libraries
import numpy as np
import os as os
from glob import glob
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import sonde_reanalysis_comparison as src
import igra_utils
import igra
import csv

### User inputs ###

# Paths and filenames
local_path = './raw_data/' # 'data/'
station_fullpath = 'data/igra2-station-list.txt'
ignore_filename = 'ignore_list_12to24_V1.csv'
load_fullpath = './file_to_load_here.pkl' # igra_dataframe_12to16_US_V1.pkl'
save_fullpath = './Data/IGRA/igra_dataframe_23to24_30_60_global.pkl'
store_limit = 100 # Save data every N stations
start_year = 2023
end_year = 2024

# World region
region = 'World' # Choice of 'World', 'US', 'EU', 'Asia'

### Simulation set up ###

# Get list of stations
stations = igra_utils.download_station_data()
stations = igra.read.stationlist( station_fullpath )

# Stations active between 2012 - 2024
index = (stations['end']>=start_year) & (stations['start']<=end_year)

# Clean up
active_stations = stations.loc[ index ]
active_stations['wmo'] = pd.to_numeric( active_stations['wmo'] )
active_stations.reset_index( inplace=True )

# Load in WMO station information
wmo_info = pd.read_excel( 'WMO_sondetype.xls', 'RaRaRawGST' )
wmo_info = wmo_info[ wmo_info[ 'instrument' ].str.contains( 'RS92' ) ]
wmo_info['station_index_numeric'] = pd.to_numeric( wmo_info['station_index'], errors='coerce' )

# Merge stations with wmo info
active_stations = pd.merge( left=active_stations, right=wmo_info, how='left', left_on='wmo', right_on='station_index_numeric' )
active_stations.dropna( subset=['instrument'], inplace=True )
active_stations.reset_index( drop=True, inplace=True )
active_stations.set_index( 'id', inplace=True )

# Load ignore_list
if os.path.exists( ignore_filename ):
    print( 'Loading ignore list...' )
    with open( ignore_filename, newline='' ) as f:
        data = csv.reader(f)
        ignore_list = list( data )[0]
else:
    ignore_list = []

# Load if dataframe saved before
if os.path.exists( load_fullpath ):
    print( 'Loading existing dataframe...' )
    active_data = pd.read_pickle( load_fullpath )
else:
    active_data = None

# Get only active stations in region we care about
if region == 'World':
    LON_EXTENT = [ -200, 200 ] # [ -30, 50 ] # [ -135, -65 ]
    LAT_EXTENT = [ 30, 60 ] # [ -200, 200 ] [ 30, 70 ] # [ 10, 50 ]
elif region == 'US':
    LON_EXTENT = [ -140, -40 ]
    LAT_EXTENT = [ 10, 70 ]
elif region == 'EU':
    LON_EXTENT = [ -200, 200 ] # [ -30, 50 ] # [ -135, -65 ]
    LAT_EXTENT = [ -200, 200 ] # [ 30, 70 ] # [ 10, 50 ]
elif region == 'Asia':
    LON_EXTENT = [ -200, 200 ] # [ -30, 50 ] # [ -135, -65 ]
    LAT_EXTENT = [ -200, 200 ] # [ 30, 70 ] # [ 10, 50 ]
else:
    raise Exception( 'Desired region (%s) does not exist' %( region ) )

### Simulation ###

# Loop over stations and figure out number of sondes within date range
if active_data is None:
    last_save = 0
else:
    last_save = len(active_data.location.unique())
ii = last_save

print( 'Total number of stations: %d' %(len(active_stations)) )
print( 'Starting number: %d' %(ii) )
for station_id, row in active_stations.iterrows():
    
    # Break early if needed
    if ii>=10:
        break
    
    # Save early if needed
    if ii >= last_save + store_limit:
        print( 'Saving data!!!' )
        active_data.to_pickle( save_fullpath )
        # Save ignore_list as well
        if os.path.exists( ignore_filename ):
            os.remove( ignore_filename )
        with open( ignore_filename, 'w', newline='') as myfile:
             wr = csv.writer( myfile, quoting=csv.QUOTE_ALL )
             wr.writerow( ignore_list )
        last_save = ii
        # break
    
    # Station filename to download
    filename = '%s*.zip' %( station_id )
    local_fullpath = '%s/%s' %( local_path, filename )
    # print(filename, row['name'])
    
    # Skip if station in ignore_list
    if station_id in ignore_list:
        print('Skipping... %s' %(station_id))
        continue
    
    # Check if station already exists in active_data
    if active_data is not None:
        if (active_data['location']==station_id).any():
            print('Case #%d: Data for %s exists... Skipping' %(ii+1, row['name']))
            ii+=1
            continue
    
    # Check location
    if (row['lat']>=LAT_EXTENT[0]) & (row['lat']<=LAT_EXTENT[1]) & (row['lon']>=LON_EXTENT[0]) & (row['lon']<=LON_EXTENT[1]):
        print('Case #%d: Working on %s...' %(ii+1, row['name']))
    else:
        continue
    
    # Download data if it doesn't exist
    if not glob( local_fullpath ):
        print('Downloading data for %s' %(row['name']))
        print( station_id, local_path )
        igra.download.station( station_id, local_path ) # 'data/' )
    
    # Load data
    local_fullpath = glob( local_path + '/' + station_id + '*.zip' )[0]
    data, stat = igra.read.ascii_to_dataframe( local_fullpath )
    
    # Only select data for specific years
    years = data.index.year
    active_mask = (years>=2023) & (years<=2024)
    main_data = data.loc[active_mask]
    main_data['location'] = station_id # row['name'] --> name is not necessarily unique but id is
    
    # Reset so that date becomes a column
    main_data = main_data.reset_index()
    
    # Check if rhumi or dewpoint depression always null
    rhumi_always_null = main_data['rhumi'].isnull().all()
    dpd_always_null = main_data['dpd'].isnull().all()
    
    # Decision point if nulls
    if not dpd_always_null: # Use the dew point depression to begin with
        # Drop nan rows and set RH value to this column
        main_data.dropna( subset=['dpd'], inplace=True )
        main_data['RH'] = src.dpd2RH( main_data['dpd'].values, main_data['temp'].values )
        ii += 1
    elif not rhumi_always_null:
        # Drop nan rows and set RH value to this column
        main_data.dropna( subset=['rhumi'], inplace=True )
        main_data['RH'] = main_data['rhumi']
        ii += 1
    else:
        print('All nulls... adding to ignore_list')
        # Add to ignore_list
        ignore_list.append( station_id )
        continue
    
    # Convert units for ease of use
    main_data['pres'] = main_data['pres']/100 # Pa to hPa
    main_data['temp'] = main_data['temp'] + 273.15 # C to K
    
    # Add RHi and SAC and altitude information
    main_data['RHi'] = src.RHw2RHi( main_data['RH'], main_data['temp'] ) # 1*(main_data['rhumi']>=60)
    main_data['SAC'] = src.SAC_general( main_data['RH'], main_data['temp'], main_data['pres'] )
    main_data['alt'] = src.P2Alt( main_data['pres'], main_data['temp'] )
    
    # Apply correction algorithm to IGRA data
    main_data[ 'Tcorr' ], main_data[ 'RHcorr' ] = \
                 igra_utils.radcorr( row['lat'], row['lon'], main_data['date'].values[0],\
                 main_data['pres'].values, main_data['temp'].values, main_data['RH'].values )
    main_data[ 'RHicorr' ] = src.RHw2RHi( main_data['RHcorr'], main_data['Tcorr'] )
    main_data[ 'SACcorr' ] = src.SAC_general( main_data['RHcorr'], main_data['Tcorr'], main_data['pres'] )
    
    if active_data is None:
        active_data = main_data.copy()
    else:
        active_data = pd.concat( [active_data, main_data], ignore_index=True )
        # Convert types to save memory
        active_data[ 'location' ] = active_data[ 'location' ].astype('category')
        active_data[ 'SAC' ] = active_data[ 'SAC' ].astype('bool')
    
### Simulation clear up    

# Convert types to save memory
active_data[ 'location' ] = active_data[ 'location' ].astype('category')
active_data[ 'SAC' ] = active_data[ 'SAC' ].astype('bool')

# Save as pickle
print( 'Completed simulation - saving data and ignore_filename...' ) 
active_data.to_pickle( save_fullpath )

# Save ignore_list
if os.path.exists( ignore_filename ):
    os.remove( ignore_filename )
with open( ignore_filename, 'w', newline='') as myfile:
     wr = csv.writer( myfile, quoting=csv.QUOTE_ALL )
     wr.writerow( ignore_list )

print( 'Simulation complete!' )


Download complete, reading table ...
Data read from: data/igra2-station-list.txt
Data processed 2878
Data read from: data/igra2-station-list.txt
Data processed 2878
Loading ignore list...
Total number of stations: 313
Starting number: 0
Case #1: Working on BECHAR...


/tmp/ipykernel_352605/1274280275.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  active_stations['wmo'] = pd.to_numeric( active_stations['wmo'] )
/tmp/ipykernel_352605/1274280275.py:156: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_data['location'] = station_id # row['name'] --> name is not necessarily unique but id is


Case #2: Working on LINZ/HOERSCHING-FLUGHAFEN...


/tmp/ipykernel_352605/1274280275.py:156: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_data['location'] = station_id # row['name'] --> name is not necessarily unique but id is


Case #3: Working on WIEN/HOHE WARTE...


...

Case #9: Working on YARMOUTH UA...


/tmp/ipykernel_352605/1274280275.py:156: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_data['location'] = station_id # row['name'] --> name is not necessarily unique but id is
/home/chinahg/.conda/envs/contrails/lib/python3.9/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/chinahg/.conda/envs/contrails/lib/python3.9/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Case #10: Working on MANIWAKI UA...


/tmp/ipykernel_352605/1274280275.py:156: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_data['location'] = station_id # row['name'] --> name is not necessarily unique but id is
/home/chinahg/.conda/envs/contrails/lib/python3.9/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/chinahg/.conda/envs/contrails/lib/python3.9/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Completed simulation - saving data and ignore_filename...
Simulation complete!


In [3]:
# Load data
# Open the file in binary mode 
with open('/home/chinahg/GCresearch/contrailuncertainty/IGRA_processing/Data/IGRA/igra_dataframe_23to24_30_60_global.pkl', 'rb') as file: 
      
    # Call load method to deserialze 
    myvar = pickle.load(file) 
  
    print(myvar) 

                      date   pres      gph    temp  rhumi   dpd  windd  winds  \
0      2023-01-01 00:00:00  936.0      NaN  282.35    NaN  15.0   50.0    2.1   
1      2023-01-01 00:00:00  927.0      NaN  284.75    NaN  16.0    NaN    NaN   
2      2023-01-01 00:00:00  925.0    902.0  284.75    NaN  16.0   70.0    4.6   
3      2023-01-01 00:00:00  900.0      NaN  283.95    NaN  16.0    NaN    NaN   
4      2023-01-01 00:00:00  860.0      NaN  281.55    NaN  14.0    NaN    NaN   
...                    ...    ...      ...     ...    ...   ...    ...    ...   
422853 2024-02-18 12:00:00   10.0  30570.0  219.25    NaN  35.0    5.0   14.4   
422854 2024-02-18 12:00:00    8.9      NaN  223.25    NaN  37.0    NaN    NaN   
422855 2024-02-18 12:00:00    7.2      NaN  219.85    NaN  34.0    NaN    NaN   
422856 2024-02-18 12:00:00    6.3      NaN  224.45    NaN  38.0    NaN    NaN   
422857 2024-02-18 12:00:00    5.2      NaN  218.25    NaN  33.0    NaN    NaN   

           location        

In [4]:
# Selected data only in 30-60 degree latitude range 
# 30-60 degrees is where most commercial aviation occurs (cite) and contains 2-3 jet streams

# Now we want to average all RHi for a single latitude, to produce one RHi profile

# 1) Pick out profiles where RHi >= 100 only 
bins = []
for i in range(len(myvar.alt)-1):
    if myvar.alt[i+1] < myvar.alt[i]: # Switches from low to high value
        # Record bin sizes
        bins.append(i)
        
# Sort data into separate arrays
j = 0
bin_bool = np.zeros_like(bins)
for j in range(len(bins)):
    i = 0
    SAC_bool = 0
    for i in range(bins[j]):
        if myvar.SAC[i] == True:
            SAC_bool = 1
    if SAC_bool == 1:
        bin_bool = 1
print("done!")
# 2) Sort data into bins (bin 1 = data for altitudes 0-1km)


# 3) Average values in each bin


# 3) 

done!


In [ ]:
plt.xlabel("RHi [%]")
plt.ylabel("Pressure [hPa]")
plt.plot(100*np.ones(45), myvar.pres[333:378], linestyle='dashed')
plt.plot(myvar.RHicorr[333:378], myvar.pres[333:378])
plt.plot(myvar.RHicorr[362]*myvar.SACcorr[362], myvar.pres[362], linestyle='dotted', marker='o', color='b')